# Scientific paper corpus builder

Builds a DataFrame of **open-access** scientific papers across all fields with:
`authors, title, content (full text), date, country, journal, theme`.

Source: **OpenAlex** (legal OA metadata + full-text links) -> download repository-hosted PDF -> extract text with PyMuPDF -> tag a custom `theme` with Claude.

All logic lives in `pipeline.py`; this notebook just drives it.

In [ ]:
import importlib, pipeline
importlib.reload(pipeline)  # re-run after editing pipeline.py
from pipeline import build_corpus, fetch_metadata, download_fulltext

## 1. Metadata only (no downloads, no API key)
Fast sanity check that the OpenAlex query returns what you expect.

In [ ]:
import pandas as pd
papers = list(fetch_metadata(20))  # all fields, OA + full text available
pd.DataFrame([{'title': p.title, 'country': p.country, 'journal': p.journal,
               'date': p.date, 'n_pdf': len(p.pdf_candidates)} for p in papers])

## 2. Custom theme tagging needs a Claude API key
Get one at https://console.anthropic.com -> API Keys, then set it below (or in your shell as `ANTHROPIC_API_KEY`).

In [ ]:
import os
os.environ['ANTHROPIC_API_KEY'] = ''  # <-- paste your key here, or leave blank and set with_theme=False

## 3. Full build: metadata + full text + theme
Start small to validate, then scale `n` up. Tune the corpus with `search=` and `extra_filters=`.

In [ ]:
df = build_corpus(
    n=25,
    search=None,                 # e.g. 'machine learning' to focus a topic; None = all fields
    extra_filters='from_publication_date:2022-01-01',  # recent papers = better OA coverage
    with_fulltext=True,
    with_theme=bool(os.environ.get('ANTHROPIC_API_KEY')),
)
df[['title', 'authors', 'country', 'journal', 'date', 'theme']]

## 4. Inspect & save

In [ ]:
print('rows:', len(df))
print('with full text:', df['content'].notna().sum())
print('themes:', df['theme'].dropna().value_counts().to_dict() if 'theme' in df else 'n/a')
df.to_parquet('papers.parquet')   # keeps lists/long text cleanly
df.to_csv('papers.csv', index=False)
print('saved papers.parquet + papers.csv')

## Scaling to a large corpus
- Bump `n` (OpenAlex paginates via cursor automatically). Tens of thousands is fine.
- Keep your `MAILTO` set in `pipeline.py` for the polite (faster) pool.
- For *very* large pulls, use the OpenAlex full snapshot (S3) instead of the API.
- Run `download_fulltext` in a thread pool to parallelize PDF fetching.
- `with_theme=True` calls Claude once per paper (Haiku, cheap). Batch with the Message Batches API for big runs.